This code is to clean the raw data include null and short 

In [1]:
import pandas as pd
import os
import glob
from pathlib import Path

# 
#  Find the repo root
#
NOTEBOOK_DIR = Path.cwd()

def find_repo_root(start_path: Path):
    cur = start_path
    for _ in range(10):
        if (cur / ".git").exists() or (cur / "data").exists():
            return cur
        cur = cur.parent
    return start_path

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

# 
# input/output path
# 
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
csv_path = RAW_DIR / "train.csv"

# 
# middle chunks path
#
out_dir1 = PROC_DIR / "clean_chunks22"
out_dir1.mkdir(parents=True, exist_ok=True)

# Final output parquet
final_output_path = PROC_DIR / "final_processed_data.parquet"

print("Repo root:", REPO_ROOT)
print("CSV path :", csv_path)
print("Chunk dir:", out_dir1)
print("Final out:", final_output_path)

# 
# Read cols name
#
cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
print("Total columns:", len(cols))
print(cols)

#
# Read chunks and Drop null and save parquet
#
chunksize = 200000

chunk_id = 0
total_rows = 0
total_rows_after = 0

reader = pd.read_csv(csv_path, chunksize=chunksize)

for chunk in reader:
    chunk_id = chunk_id + 1

    rows_before = len(chunk)
    total_rows = total_rows + rows_before

    chunk_clean = chunk.dropna()

    rows_after = len(chunk_clean)
    total_rows_after = total_rows_after + rows_after

    dropped_count = rows_before - rows_after

    if rows_before > 0:
        drop_rate = (dropped_count / rows_before) * 100
    else:
        drop_rate = 0.0

    print(f"\n[Chunk {chunk_id}]")
    print(f"  - before cleaning : {rows_before} rows")
    print(f"  - after cleaning  : {rows_after} rows")
    print(f"  - delete          : {dropped_count} rows ({round(drop_rate, 2)}%)")

    file_name = f"clean_{chunk_id}.parquet"
    out_path = out_dir1 / file_name
    chunk_clean.to_parquet(out_path, index=False)

print("\n========== Summary ==========")
print(f"total piece of chunk: {chunk_id}")
print(f"total rows before clean: {total_rows}")
print(f"total rows after clean : {total_rows_after}")

if total_rows > 0:
    overall_dropped = total_rows - total_rows_after
    overall_rate = (overall_dropped / total_rows) * 100
    print(f"total dropped ratio: {round(overall_rate, 2)}%")
else:
    print("Error: total_rows is 0")

# 
# Get all chunk parquet and drop short sequence and final save a parquet
#
all_files = glob.glob(str(out_dir1 / "*.parquet"))
list_of_dfs = [pd.read_parquet(filename) for filename in all_files]
col_all = pd.concat(list_of_dfs, ignore_index=True)

seq_lengths = col_all.groupby("sequence_id").size()
mean_len = seq_lengths.mean()
std_len = seq_lengths.std()

print(f"\naverage sequence length : {mean_len:.2f}")
print(f"sequence length std      : {std_len:.2f}")

threshold = mean_len - std_len
print(f"threshod (Mean - Std)    : {threshold:.2f}")

valid_ids = seq_lengths[seq_lengths >= threshold].index
col_final = col_all[col_all["sequence_id"].isin(valid_ids)]

num_removed = len(seq_lengths) - len(valid_ids)
print(f"delete {num_removed} short sequence_id(s)")
print(f"final rows after filter  : {len(col_final)}")

col_final.to_parquet(final_output_path, index=False)
print(f"final data is save to: {final_output_path}")


Repo root: E:\A1_last_version
CSV path : E:\A1_last_version\data\raw\train.csv
Chunk dir: E:\A1_last_version\data\processed\clean_chunks22
Final out: E:\A1_last_version\data\processed\final_processed_data.parquet
Total columns: 341
['row_id', 'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5', 'tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3', 'tof_1_v4', 'tof_1_v5', 'tof_1_v6', 'tof_1_v7', 'tof_1_v8', 'tof_1_v9', 'tof_1_v10', 'tof_1_v11', 'tof_1_v12', 'tof_1_v13', 'tof_1_v14', 'tof_1_v15', 'tof_1_v16', 'tof_1_v17', 'tof_1_v18', 'tof_1_v19', 'tof_1_v20', 'tof_1_v21', 'tof_1_v22', 'tof_1_v23', 'tof_1_v24', 'tof_1_v25', 'tof_1_v26', 'tof_1_v27', 'tof_1_v28', 'tof_1_v29', 'tof_1_v30', 'tof_1_v31', 'tof_1_v32', 'tof_1_v33', 'tof_1_v34', 'tof_1_v35', 'tof_1_v36', 'tof_1_v37', 'tof_1_v38', 'tof_1_v39', 'tof_1_v40', 'tof_1_v41', 'to